# Modeling Delivery Times with Neural Networks

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/deep-learning/03-lab_nn/lab_nn_exercise.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

Welcome to the delivery challenge!

In this lab you will train a neural network to predict delivery time from distance. You will start with a **simple linear network** that works on bike-only data. Then you will see why that same model fails when the relationship **curves** — and how a **non-linear activation function** (ReLU) lets the network fit the curve.


In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/deep-learning/03-lab_nn"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)

## Imports

These will give you:
- **`pandas`**: Load tabular data from a CSV file into a DataFrame
- **`Pipeline`**: Chain preprocessing and the model so they stay in sync
- **`StandardScaler`**: Standardize features so they have mean 0 and variance 1
- **`MLPRegressor`**: A multi-layer perceptron for regression (predicting a continuous value, like delivery time)

scikit-learn also has [`MLPClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) for classification problems. This lab predicts a number (minutes), so you will use [`MLPRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html).


In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

import helper_utils

## The Delivery Problem

**Question:** Can a 7-mile delivery finish in under 30 minutes?

You have a handful of recent **bike** deliveries: distance in miles, and how long each one took. You will train a tiny neural network on that data, then use it to decide whether to take a 7-mile job.


## Linear Bike Data

Load `bike_deliveries.csv`.

* `X_bike` is a DataFrame with one column, `distance_miles`. Keeping it 2D (a table with rows and a column) is what scikit-learn expects for features.
* `y_bike` is a Series of delivery times in minutes. A 1D target is what `MLPRegressor` expects for a single output.


In [ ]:
# 1. Load bike_deliveries.csv into a DataFrame named bike_df


# 2. Select the distance column as a 2D DataFrame named X_bike


# 3. Select the delivery-time column as a 1D Series named y_bike


bike_df

* These four points follow a **straight line**: each extra mile adds about the same amount of time.

In [ ]:
helper_utils.plot_data(X_bike, y_bike, title="Bike Delivery Data")

## A Simple Linear Neural Network

A single linear neuron predicts `time ≈ weight × distance + bias`.

In scikit-learn, that is an `MLPRegressor` with:

* **`hidden_layer_sizes=()`**: no hidden layer — just input to output, one linear neuron
* **`activation="identity"`**: no extra non-linearity (the output is a straight line)
* **`solver="lbfgs"`**: a good default on tiny datasets
* **`random_state=42`**: reproducible weight initialization

You are **not** scaling the features yet. With one feature and a handful of miles, you can read the learned weight as "minutes per mile" and the bias as fixed overhead (pickup, start).


In [ ]:
# 1. Create an MLPRegressor named linear_model:
#    hidden_layer_sizes=(), activation="identity", solver="lbfgs",
#    max_iter=1000, random_state=42
linear_model = MLPRegressor(
    # your code here
)

## Training

Training is a single call: `linear_model.fit(X_bike, y_bike)`.

Behind the scenes, `MLPRegressor` minimizes **squared error** — how far the predicted times are from the actual times — and updates the weight and bias. You do not write a loop over epochs or step an optimizer yourself.


In [ ]:
# 1. Fit linear_model on X_bike and y_bike


print("Training complete.")

### Visualize

Plot the data points against the learned line:


In [ ]:
helper_utils.plot_fit(
    linear_model,
    X_bike,
    y_bike,
    title="Linear Model Fit",
    pred_label="Predicted Line",
)

### Predict

Use the trained model to predict time for a new distance. Start with `7.0` miles.

Pass a one-row DataFrame with the same column name as training, `distance_miles`. `predict` returns an array; take the first element to get minutes.


In [ ]:
distance_to_predict = 7.0

In [ ]:
# 1. Create a one-row DataFrame named new_distance with column distance_miles


# 2. Predict and take the first element as predicted_time


print(f"Prediction for a {distance_to_predict}-mile delivery: {predicted_time:.1f} minutes")

if predicted_time > 30:
    print("\nDecision: Do NOT take the job. You will likely be late.")
else:
    print("\nDecision: Take the job. You can make it!")

A 7-mile delivery is predicted over 30 minutes — decline the job.


## Inspect Parameters

Read the learned weight and bias. With no hidden layer, `coefs_[0]` is the input-to-output weight and `intercepts_[0]` is the bias.


In [ ]:
weight = linear_model.coefs_[0].item()
bias = linear_model.intercepts_[0].item()

print(f"Weight: {weight:.2f}")
print(f"Bias: {bias:.2f}")

**Interpretation**

- **Weight (~5.0):** minutes added per mile
- **Bias (~2.0):** fixed overhead (pickup, start)

`Time ≈ 5.0 × Distance + 2.0`


## Harder Data

Deliveries over 3 miles now go by **car**. The new dataset mixes bike and car trips — does the linear model still fit?

Load `delivery_data.csv`. This is the combined bike and car data.


In [ ]:
# 1. Load delivery_data.csv into a DataFrame named df


# 2. Select the distance column as a 2D DataFrame named X


# 3. Select the delivery-time column as a 1D Series named y


df.head()

* As you will see from running the code below, this data follows a **curve**, not a straight line.

In [ ]:
helper_utils.plot_data(X, y, title="Delivery Data (Bikes & Cars)")

Predict on the combined data with the **already-trained linear model** — do not retrain yet. Then plot the mismatch.


In [ ]:
helper_utils.plot_fit(
    linear_model,
    X,
    y,
    title="Linear Model vs. Non-Linear Reality",
    pred_label="Linear Model Predictions",
)

**Why it fails:** bike+car data is non-linear. The first few miles beyond bike range hit dense city traffic. Further out, cars reach highways and move faster. A straight line cannot fit both regimes.

A linear model treats every mile as needing the same amount of time. Reality **curves**.


## Why Adding More Neurons Isn't Enough

You might think: "Let's just add more neurons. If one neuron learns one relationship, maybe multiple neurons can capture more complexity."

But here's the problem: **Simply adding more linear neurons is not the solution. The model's output would still be a straight line.**


![](../assets/two_neurons_still_linear.png)


When you have just one neuron, it produces your final prediction, so it is the output layer. But when you add a second neuron, now you have two outputs from your single input, and you need a single prediction in your final output. So you add another neuron to combine these two outputs into one final output, and now those two neurons are considered a **hidden layer**.

These neurons independently calculate the weighted sum from the input. But here's the problem: **That's still just a linear equation no matter how many neurons you stack like this.** If all operations on a single input are linear—simply multiplying by weights and adding biases—you'll always end up with a straight line.

> Try it out yourself! You can experiment with adding more neurons here: [https://playground.tensorflow.org/](https://playground.tensorflow.org/#activation=linear&batchSize=10&dataset=circle&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4,2&seed=0.21856&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false) where we set the **Activation function to: `linear`** (i.e., no activation function).

## The Solution: Activation Functions

To learn curves and not just straight lines, your model needs something more than a linear transformation. It needs **non-linear activation functions**. These functions add just enough complexity to help the model learn more interesting patterns.

A neuron starts by computing a linear transformation of its input. But instead of sending that raw number straight out, we pass it through an activation function. **This small change, adding a nonlinear transformation, is what lets your model learn much richer patterns.**


![](../assets/activation_after_neuron.png)


## Introducing ReLU

In this lab, you'll use the most popular and powerful activation function, **ReLU (Rectified Linear Unit)**. It's incredibly simple and surprisingly powerful.

Here's what ReLU does:
- **If the input is negative, output 0**
- **If the input is positive, just pass it through unchanged**


In [ ]:
def my_relu(input: float):
    # 1. Return input if positive, otherwise return 0


assert my_relu(-2.4) == 0
assert my_relu(3.6) == 3.6

![](../assets/relu.png)


It's as simple as that! In scikit-learn, you set `activation='relu'` on the MLP, and ReLU is applied to every hidden neuron.

## How ReLU Creates Curves

When you add ReLU to a single neuron, your neuron will start by computing `W × Distance + B`. Without ReLU, this will give you a straight line. But with ReLU:
- When `W × Distance + B` is **negative**, the output is **0**, which is a flat line
- When it's **positive**, the output follows the line normally

**You've just escaped the world of straight lines. You now have a bend, a corner where the behavior changes.**

The bend occurs where `W × Distance + B = 0`, which effectively gives us `Distance = -B/W`—where it stops outputting 0 and starts responding to the input.


## Multiple Neurons, Multiple Bends

Think about your city transportation data. The pattern doesn't just bend once. It curves, shifting gradually across different distances as traffic conditions change. **So if one neuron gives us one bend, then maybe multiple neurons will give us multiple bends.**

Each neuron has its own weight and bias, and that means that each neuron activates at a different distance:
- Neuron 1 might activate right around 3 miles where the traffic starts
- Neuron 2 might activate right around 8 miles as we're entering the highways
- Neuron 3 might activate at around 15 miles when we're at full highway speeds

When you add their outputs together, their combined output approximates your complex curve. **With enough neurons, each learning where to activate and how strongly to contribute, you can approximate any smooth curve.**


![](../assets/five_relu_activated_neurons.png)


## Building a Non-Linear Model

You will now:

* Put a `StandardScaler` inside a scikit-learn **pipeline** so feature scaling happens automatically during training and prediction.
* Build a *non-linear* neural network with `MLPRegressor` using the **ReLU** activation function and **3 hidden neurons**.
* Train it on the combined bike and car data.
* Predict delivery times and see if it can succeed where the linear model failed.


### Scaling Features in a Pipeline

Neural networks are sensitive to the scale of input features, so scikit-learn recommends scaling them before training an MLP — especially once distances range from 1 to 20 miles.

#### Why Scale?

When features have very different scales, training can become unstable. Large values can dominate the learning process. Standardization helps by:

- **Preventing large distance values from dominating the learning process**
- **Keeping the optimization stable**: when values are on similar scales, weight updates stay more balanced

#### Standardization (Z-Score Normalization)

Standardization converts each feature onto a new scale where:
- The mean becomes 0
- The standard deviation becomes 1
- Values are typically in the range of approximately -3 to +3

The underlying curved pattern of the data stays the same — only the axis scale changes.

#### Why a Pipeline?

You *could* scale the distances by hand with `normalized_value = (value - mean) / standard_deviation`. That is easy to get wrong at prediction time: you must reuse the **training** mean and standard deviation, not compute new ones from the new input.

A scikit-learn **pipeline** does this for you. You will put a `StandardScaler` in front of `MLPRegressor`. When you call `fit`, the scaler learns the mean and standard deviation from the training distances and then trains the MLP on the scaled values. When you later call `predict`, the same fitted scaler is applied to the new distance automatically.


### Understanding the Architecture

In scikit-learn, you describe the network with a few arguments. The input is just your data. The output layer is created for you. ReLU is the activation applied to each hidden neuron.

* **`hidden_layer_sizes=(3,)`**: This is your **hidden layer**.
    * The tuple `(3,)` means one hidden layer with three neurons
    * Each of these three neurons independently calculates a weighted sum from the (scaled) input: `W × Distance + B`
    * Each neuron has its own weight and bias, which means each neuron will activate at a different distance
    * If you wanted two hidden layers, you would write something like `(8, 4)` — eight neurons, then four

* **`activation='relu'`**: This applies the ReLU activation function to the output of each hidden neuron.
    * This is the crucial non-linear step that allows your model to create "bends" and learn curves instead of just straight lines
    * Each neuron's output passes through ReLU: if the value is negative, it becomes 0; if positive, it passes through unchanged

* **Output layer**: `MLPRegressor` adds this for you.
    * It takes the three activated values from the hidden layer as its input
    * It combines them with a single neuron to produce one final output: the predicted delivery time in minutes
    * The output uses the identity activation (no extra non-linearity), which is what you want for regression
    * The loss is squared error, the same idea as the linear model earlier

* **`solver='lbfgs'`**: With only a few dozen training rows, scikit-learn recommends L-BFGS over Adam or SGD. It often converges faster and more reliably on small datasets.

### How It All Works Together

This creates a neural network with **1 hidden layer containing 3 neurons**. Here's the flow:

1. **Input**: Distance in miles (the pipeline scales it first)
2. **Hidden Layer**: Three neurons each compute `W × Distance + B`, creating three intermediate values
3. **ReLU Activation**: Each of the three values passes through ReLU, creating potential "bends" at different activation points
4. **Output Layer**: The three activated values are combined into a single prediction

If you want a smoother curve, just increase the number of neurons in `hidden_layer_sizes`.


![](../assets/three_relu_activated_neurons.png)


In [ ]:
# 1. Build a Pipeline: StandardScaler, then MLPRegressor
#    with hidden_layer_sizes=(3,), activation="relu", solver="lbfgs",
#    max_iter=3000, and random_state=287
model = Pipeline([
    # your code here
])

## Training the Non-Linear Model

With the pipeline defined, training is a single call: `model.fit(X, y)`.

Behind the scenes, the pipeline first fits `StandardScaler` on the distances, transforms them, then trains the MLP. Squared error is the default loss, and `random_state=287` keeps the weight initialization reproducible.

`max_iter=3000` is a cap on how long L-BFGS may run. On this small dataset it will usually stop earlier, once the loss has stopped improving.


In [ ]:
# 1. Fit the pipeline on X and y


print("Training complete.")

## Checking the Final Fit

Plot your model's predicted curve against the original data points. This lets you inspect how well the non-linear model learned the complex pattern. Pass `mark_bends=True` so dashed lines mark where each hidden neuron bends.


In [ ]:
helper_utils.plot_fit(
    model,
    X,
    y,
    title="Non-Linear Model Fit vs. Actual Data",
    pred_label="Non-Linear Model Predictions",
    mark_bends=True,
)

<br>

Congratulations! You have trained both a **linear** network and a **non-linear** network with scikit-learn.

Where the simple linear model failed on bike+car data, the ReLU model succeeds. It learned to capture the curved relationship.


## Making a Prediction

Because the scaler lives **inside** the pipeline, you pass the distance in **miles**. The pipeline applies the same standardization it learned during `fit`, then the MLP predicts delivery time in minutes.

The company now promises deliveries within **45 minutes** (longer mixed bike/car trips take more time than the 30-minute bike-only jobs) and wants to know which vehicle to use.


In [ ]:
distance_to_predict = 5.1

In [ ]:
# 1. Create a one-row DataFrame named new_distance with column distance_miles


# 2. Predict with model and take the first element as predicted_time


print(f"Prediction for a {distance_to_predict}-mile delivery: {predicted_time:.1f} minutes")

if predicted_time > 45:
    print("\nDecision: Do NOT promise the delivery in under 45 minutes.")
else:
    if distance_to_predict <= 3:
        print(f"\nDecision: Yes, delivery is possible. Since the distance is {distance_to_predict} miles (<= 3 miles), use a bike.")
    else:
        print(f"\nDecision: Yes, delivery is possible. Since the distance is {distance_to_predict} miles (> 3 miles), use a car.")

## Conclusion

You started with a one-neuron linear network that fit bike-only data and made a 7-mile decision. That same model failed when cars were added, because a straight line cannot follow a curve.

Adding a non-linear activation function like **ReLU**, plus a few hidden neurons, gave the model the ability to succeed. You also put a `StandardScaler` in a **pipeline** so feature scaling stays attached to the model at training and prediction time.


### About Activation Functions

We've been using ReLU because it's the workhorse of modern deep learning. It's fast, it's effective, and it's widely used. But it's not the only activation function out there:

- **Sigmoid**: Squashes your outputs into a range between 0 and 1, great for probabilities
- **Tanh**: Maps your values to a range between -1 and +1, really useful for many tasks
- And there are lots of others, too


![](../assets/activation_functions.png)


If you want to dive deeper, see scikit-learn's [neural network models (supervised)](https://scikit-learn.org/stable/modules/neural_networks_supervised.html) guide and the [`MLPRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html) docs. `MLPRegressor` also supports `activation='logistic'` (sigmoid) and `activation='tanh'`. But honestly, for most situations, **ReLU is all you need**.


### What's Next?

With these fundamental skills of building architectures, preparing data with pipelines, and training models, you are well prepared for the next step: new kinds of problems, like classification — where you would reach for [`MLPClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) instead of `MLPRegressor` — and the mechanics of how neural networks learn.
